In [8]:
import pandas as pd
import torch

from transformers import (
    BertTokenizer,
    BertModel,
    BertForSequenceClassification
)

In [10]:
train_df = pd.read_csv("../data/raw/train.csv")
test_df = pd.read_csv("../data/raw/test.csv")

In [12]:
# Display dataset information

print("Training Set Shape :", train_df.shape)
print("Test Set Shape     :", test_df.shape)

# Preview the training dataset

train_df.head()

Training Set Shape : (7613, 5)
Test Set Shape     : (3263, 4)


,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [15]:
# Keep only the columns required for BERT

bert_df = train_df[["text", "target"]].copy()

bert_df.head()


,text,target
0,Our Deeds are the Reason of this #earthquake M...,1
1,Forest fire near La Ronge Sask. Canada,1
2,All residents asked to 'shelter in place' are ...,1
3,"13,000 people receive #wildfires evacuation or...",1
4,Just got sent this photo from Ruby #Alaska as ...,1


In [16]:
# Check for missing values

bert_df.isnull().sum()

text      0
target    0
dtype: int64

### Tokenizing Disaster Tweets

BERT cannot process raw text directly.

Each tweet must first be converted into a format that BERT understands.

The `BertTokenizer` performs the following tasks automatically:

- Splits text into WordPiece tokens
- Adds special tokens (`[CLS]` and `[SEP]`)
- Converts tokens into vocabulary IDs (`input_ids`)
- Creates an `attention_mask`
- Creates `token_type_ids`

These outputs become the inputs to the BERT model during fine-tuning.

In [17]:
# Load the Pretrained Tokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [18]:
# Display a real tweet from our dataset

sample_tweet = bert_df.loc[0, "text"]

print(sample_tweet)

Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all


In [19]:
# Tokenize the sample tweet

encoded_tweet = tokenizer(
    sample_tweet,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

In [20]:
# Display everything produced by the tokenizer

encoded_tweet

{'input_ids': tensor([[  101,  2256, 15616,  2024,  1996,  3114,  1997,  2023,  1001,  8372,
          2089, 16455,  9641,  2149,  2035,   102,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,  

In [21]:
# Display the tokenizer outputs separately

for key, value in encoded_tweet.items():
    print(f"{key}:")
    print(value)
    print("-" * 60)

input_ids:
tensor([[  101,  2256, 15616,  2024,  1996,  3114,  1997,  2023,  1001,  8372,
          2089, 16455,  9641,  2149,  2035,   102,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     

In [22]:
# Convert token IDs back into readable tokens

tokens = tokenizer.convert_ids_to_tokens(
    encoded_tweet["input_ids"][0]
)

print(tokens)

['[CLS]', 'our', 'deeds', 'are', 'the', 'reason', 'of', 'this', '#', 'earthquake', 'may', 'allah', 'forgive', 'us', 'all', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PA

In [23]:
# Count the number of real tokens before padding

attention_mask = encoded_tweet["attention_mask"][0]

real_tokens = attention_mask.sum().item()

print(f"Number of real tokens: {real_tokens}")
print(f"Maximum sequence length: 128")

Number of real tokens: 16
Maximum sequence length: 128


In [ ]:
# Load Base BERT Model

bert_model = BertModel.from_pretrained("bert-base-uncased")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [25]:
# Load BERT for Classification
classifier_model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [26]:
# Compare the two models

print(type(bert_model))
print(type(classifier_model))

<class 'transformers.models.bert.modeling_bert.BertModel'>
<class 'transformers.models.bert.modeling_bert.BertForSequenceClassification'>


In [27]:
classifier_model.classifier

Linear(in_features=768, out_features=2, bias=True)

In [30]:
outputs = classifier_model(**encoded_tweet)
print(outputs.keys())
print(outputs.logits)

odict_keys(['logits'])
tensor([[ 0.3227, -0.1392]], grad_fn=<AddmmBackward0>)


### Understanding Logits

The output of `BertForSequenceClassification` is **not** a class label or probability.

Instead, the model produces **logits**, which are raw prediction scores for each class.

During training, these logits are passed to the loss function (`CrossEntropyLoss`).

During inference, the class with the highest logit is selected as the prediction.

In [31]:
# Get predicted class

predicted_class = torch.argmax(outputs.logits, dim=1)

print(predicted_class)

tensor([0])


In [32]:
# Convert logits to probabilities

probabilities = torch.softmax(outputs.logits, dim=1)

print(probabilities)
print(probabilities.sum())

tensor([[0.6135, 0.3865]], grad_fn=<SoftmaxBackward0>)
tensor(1., grad_fn=<SumBackward0>)


In [ ]:

# Fine-Tuning Pipeline
# (Implemented in Google Colab)


# 1. Load Disaster Tweets dataset

# 2. Split into training and validation sets

# 3. Tokenize all tweets

# 4. Create PyTorch Dataset

# 5. Create DataLoader

# 6. Load BertForSequenceClassification

# 7. Define Optimizer (AdamW)

# 8. Fine-tune BERT

# 9. Evaluate on validation set

# 10. Save trained model

# 11. Load trained model in Streamlit